# Extract virus detections (multi-source)

Builds a unified raw table under `data/raw/virus/` from programmatic feeds aligned with [`specs/02-virus-dataset-selection.md`](../specs/02-virus-dataset-selection.md).

| Source key | Feed | Status in this notebook |
|------------|------|-------------------------|
| `ena_sequence` / `ena_read_run` | ENA Portal `sequence` + `read_run` + BioSample backfill | **Implemented** |
| `bvbrc` | BV-BRC Genome API | **Implemented** |
| `pathoplexus` | Pathoplexus LAPIS | Pending |
| `empres_i` | FAO EMPRES-i BigQuery CSV | Pending |
| `ecdc_atlas` | ECDC WNV weekly/monthly NUTS3 CSVs | Pending |

- Endpoints: ENA Portal `https://www.ebi.ac.uk/ena/portal/api/search`; BV-BRC `https://www.bv-brc.org/api/genome/`
- License / terms: [EMBL-EBI Terms of Use](https://www.ebi.ac.uk/about/terms-of-use/); [BV-BRC Data Usage](https://www.bv-brc.org/) (no account required)
- Shared helpers: `vector_guard.virus_extract` (`VIRUS_RAW_COLUMNS`, keep filter, later dedupe)

This notebook stops at the raw extract (TSV + manifest + smoke check). Cleaning, geocoding, and VectorNet joins come later.

In [1]:
from __future__ import annotations

import hashlib
import io
import json
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import pandas as pd
from tqdm import tqdm

from vector_guard.data import (
    VIRUS_VECTOR_TAXON_MAP,
    default_virus_raw_dir,
    find_repo_root,
    load_virus_detections,
    resolve_virus_core_path,
)
from vector_guard.virus_extract import (
    VIRUS_RAW_COLUMNS,
    apply_keep_filter,
    assign_record_ids,
    empty_virus_frame,
    ensure_virus_columns,
    lineage_tax_ids,
    normalize_accession,
    pick_primary_accession,
    resolve_coords,
    split_country_subnational,
)

REPO_ROOT = find_repo_root()
RAW_DIR = default_virus_raw_dir(REPO_ROOT)
TSV_NAME = "virus_detections.tsv"
TSV_PATH = RAW_DIR / TSV_NAME
MANIFEST_PATH = RAW_DIR / "manifest.json"

ENA_SEARCH_URL = "https://www.ebi.ac.uk/ena/portal/api/search"
ENA_CITATION = (
    "European Nucleotide Archive (ENA), EMBL-EBI. Sequence / read-run / sample "
    "metadata retrieved via the ENA Portal API (INSDC: ENA + GenBank + DDBJ). "
    "https://www.ebi.ac.uk/ena/portal/api/"
)
BVBRC_GENOME_URL = "https://www.bv-brc.org/api/genome/"
BVBRC_CITATION = (
    "Bacterial and Viral Bioinformatics Resource Center (BV-BRC). Viral genome "
    "metadata retrieved via the Genome REST API. "
    "https://www.bv-brc.org/api/genome/"
)
DOWNLOAD_TIMEOUT_S = 600
CHUNK_SIZE = 1024 * 256
USER_AGENT = "vector-guard/0.1"
MAX_RETRIES = 5
RETRY_BACKOFF_S = 2.0
SAMPLE_BATCH_SIZE = 1000
BVBRC_PAGE_SIZE = 25000

BVBRC_SELECT_FIELDS = [
    "genome_id",
    "genome_name",
    "taxon_id",
    "genbank_accessions",
    "biosample_accession",
    "bioproject_accession",
    "isolation_country",
    "geographic_location",
    "state_province",
    "latitude",
    "longitude",
    "collection_date",
    "collection_year",
    "host_name",
    "isolation_source",
]

TARGETS = VIRUS_VECTOR_TAXON_MAP

ENA_SEQUENCE_FIELDS = [
    "accession",
    "scientific_name",
    "tax_id",
    "country",
    "location",
    "collection_date",
    "host",
    "isolation_source",
    "sample_accession",
    "study_accession",
]
ENA_READ_RUN_FIELDS = [
    "run_accession",
    "sample_accession",
    "study_accession",
    "scientific_name",
    "tax_id",
    "country",
    "location",
    "lat",
    "lon",
    "collection_date",
    "host",
    "isolation_source",
]
ENA_SAMPLE_FIELDS = [
    "sample_accession",
    "study_accession",
    "scientific_name",
    "tax_id",
    "country",
    "location",
    "lat",
    "lon",
    "collection_date",
    "host",
    "isolation_source",
]

# Filled by each source section; merge/write consumes this dict later.
source_frames: dict[str, pd.DataFrame] = {}
source_stats: dict[str, dict] = {}

RAW_DIR, ENA_SEARCH_URL, BVBRC_GENOME_URL, len(TARGETS), len(VIRUS_RAW_COLUMNS)

(PosixPath('/workspaces/vector-guard/data/raw/virus'),
 'https://www.ebi.ac.uk/ena/portal/api/search',
 26,
 19)

## Wipe local raw directory

Every full extract run replaces the previous local copy.

In [2]:
def wipe_raw_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)
    for child in path.iterdir():
        if child.name == ".gitkeep":
            continue
        if child.is_dir():
            shutil.rmtree(child)
        else:
            child.unlink()
    (path / ".gitkeep").touch(exist_ok=True)


wipe_raw_dir(RAW_DIR)
print(f"Wiped and recreated: {RAW_DIR}")

Wiped and recreated: /workspaces/vector-guard/data/raw/virus


## Shared HTTP helpers

Retrying ENA Portal GET/POST helpers used by the sequence, read-run, and sample steps.

In [3]:
def _http_body(
    request: Request,
    *,
    timeout_s: int = DOWNLOAD_TIMEOUT_S,
    retries: int = MAX_RETRIES,
) -> bytes:
    last_error: Exception | None = None
    for attempt in range(retries):
        try:
            with urlopen(request, timeout=timeout_s) as response:
                chunks: list[bytes] = []
                while True:
                    chunk = response.read(CHUNK_SIZE)
                    if not chunk:
                        break
                    chunks.append(chunk)
            return b"".join(chunks)
        except HTTPError as exc:
            last_error = exc
            # Do not retry client errors (e.g. invalid sample accessions → 400).
            if 400 <= int(exc.code) < 500:
                break
            if attempt + 1 >= retries:
                break
            time.sleep(RETRY_BACKOFF_S * (2**attempt))
        except (URLError, TimeoutError) as exc:
            last_error = exc
            if attempt + 1 >= retries:
                break
            time.sleep(RETRY_BACKOFF_S * (2**attempt))
    assert last_error is not None
    raise last_error


def ena_search_tsv(
    *,
    result: str,
    query: str,
    fields: list[str],
    timeout_s: int = DOWNLOAD_TIMEOUT_S,
) -> pd.DataFrame:
    """GET ENA Portal search results as a string-typed DataFrame."""
    params = {
        "result": result,
        "query": query,
        "fields": ",".join(fields),
        "format": "tsv",
        "limit": "0",
    }
    url = f"{ENA_SEARCH_URL}?{urlencode(params)}"
    request = Request(url, headers={"User-Agent": USER_AGENT})
    text = _http_body(request, timeout_s=timeout_s).decode("utf-8", errors="replace")
    if not text.strip():
        return pd.DataFrame(columns=fields)
    return pd.read_csv(io.StringIO(text), sep="\t", dtype=str, keep_default_na=False)


import re

_SAMPLE_ACCESSION_RE = re.compile(
    r"^(?:SAMN\d+|SAMEA\d+|SAMD\d+|ERS\d+|SRS\d+|DRS\d+)$",
    re.IGNORECASE,
)


def _post_sample_batch(
    batch: list[str],
    *,
    fields: list[str],
    timeout_s: int,
) -> pd.DataFrame:
    """POST one BioSample accession batch; binary-split on HTTP 400 invalid IDs."""
    if not batch:
        return pd.DataFrame(columns=fields)
    data = urlencode(
        {
            "result": "sample",
            "includeAccessions": ",".join(batch),
            "fields": ",".join(fields),
            "format": "tsv",
            "limit": "0",
        }
    ).encode("utf-8")
    request = Request(
        ENA_SEARCH_URL,
        data=data,
        headers={
            "User-Agent": USER_AGENT,
            "Content-Type": "application/x-www-form-urlencoded",
        },
    )
    try:
        text = _http_body(request, timeout_s=timeout_s).decode("utf-8", errors="replace")
    except HTTPError as exc:
        if exc.code != 400 or len(batch) == 1:
            if exc.code == 400 and len(batch) == 1:
                return pd.DataFrame(columns=fields)
            raise
        mid = len(batch) // 2
        left = _post_sample_batch(batch[:mid], fields=fields, timeout_s=timeout_s)
        right = _post_sample_batch(batch[mid:], fields=fields, timeout_s=timeout_s)
        parts = [p for p in (left, right) if not p.empty]
        return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=fields)

    if not text.strip():
        return pd.DataFrame(columns=fields)
    return pd.read_csv(io.StringIO(text), sep="\t", dtype=str, keep_default_na=False)


def ena_sample_by_accessions(
    accessions: list[str],
    *,
    fields: list[str] = ENA_SAMPLE_FIELDS,
    batch_size: int = SAMPLE_BATCH_SIZE,
    timeout_s: int = DOWNLOAD_TIMEOUT_S,
) -> pd.DataFrame:
    """POST ENA Portal ``includeAccessions`` lookups for BioSamples."""
    unique = sorted(
        {
            a.strip()
            for a in accessions
            if str(a).strip() and _SAMPLE_ACCESSION_RE.match(str(a).strip())
        }
    )
    if not unique:
        return pd.DataFrame(columns=fields)

    frames: list[pd.DataFrame] = []
    for start in tqdm(range(0, len(unique), batch_size), desc="ENA samples", leave=False):
        batch = unique[start : start + batch_size]
        part = _post_sample_batch(batch, fields=fields, timeout_s=timeout_s)
        if not part.empty:
            frames.append(part)

    if not frames:
        return pd.DataFrame(columns=fields)
    out = pd.concat(frames, ignore_index=True)
    if "sample_accession" in out.columns:
        out = out.drop_duplicates(subset=["sample_accession"], keep="first")
    return out


print("ENA helpers ready")

ENA helpers ready


## ENA — `sequence` + `read_run` + sample backfill

For each pathogen in `VIRUS_VECTOR_TAXON_MAP`:

1. Fetch assembled sequences (`result=sequence`) and raw runs (`result=read_run`) via `tax_tree(<tax_id>)`.
2. Collect BioSample accessions and backfill missing place / time / host fields (`coord_source=sample_backfill` when coords come only from the sample).
3. Normalize into `VIRUS_RAW_COLUMNS` and apply the place-or-time keep filter.

Output: `ena_df` registered under `source_frames["ena"]`.

In [4]:
def _blank(series: pd.Series | None, index: pd.Index) -> pd.Series:
    if series is None:
        return pd.Series("", index=index, dtype=str)
    return series.fillna("").astype(str)


def _fill_blank(base: pd.Series, donor: pd.Series) -> pd.Series:
    base_clean = base.fillna("").astype(str).str.strip()
    donor_clean = donor.fillna("").astype(str).str.strip()
    return base_clean.where(base_clean != "", donor_clean)


def backfill_from_samples(df: pd.DataFrame, samples: pd.DataFrame) -> pd.DataFrame:
    """Fill missing ENA row fields from BioSample metadata keyed by sample_accession."""
    out = df.copy().reset_index(drop=True)
    if out.empty or samples.empty or "sample_accession" not in out.columns:
        return out

    sample_map = samples.drop_duplicates(subset=["sample_accession"], keep="first").set_index(
        "sample_accession"
    )
    keyed = out["sample_accession"].fillna("").astype(str).str.strip()
    matched = keyed.isin(sample_map.index)
    if not matched.any():
        return out

    # Align BioSample rows to the detection frame by position (same length/order).
    sample_rows = sample_map.reindex(keyed.to_numpy())
    sample_rows.index = out.index

    for column in (
        "country",
        "collection_date",
        "host",
        "isolation_source",
        "study_accession",
        "location",
    ):
        if column not in out.columns:
            out[column] = ""
        donor = _blank(sample_rows.get(column), out.index)
        out[column] = _fill_blank(_blank(out[column], out.index), donor)

    if "lat" not in out.columns:
        out["lat"] = ""
    if "lon" not in out.columns:
        out["lon"] = ""
    if "coord_source" not in out.columns:
        out["coord_source"] = "none"

    sample_lat = _blank(sample_rows.get("lat"), out.index)
    sample_lon = _blank(sample_rows.get("lon"), out.index)
    sample_location = _blank(sample_rows.get("location"), out.index)

    for idx in out.index[matched.to_numpy()]:
        current_lat = str(out.at[idx, "lat"]).strip()
        current_lon = str(out.at[idx, "lon"]).strip()
        if current_lat and current_lon:
            continue
        lat, lon, coord_source = resolve_coords(
            lat=sample_lat.at[idx],
            lon=sample_lon.at[idx],
            location=sample_location.at[idx],
            coord_source="sample_backfill",
        )
        if lat and lon:
            out.at[idx, "lat"] = lat
            out.at[idx, "lon"] = lon
            out.at[idx, "coord_source"] = coord_source

    return out

def normalize_ena_rows(
    df: pd.DataFrame,
    *,
    source: str,
    accession_column: str,
) -> pd.DataFrame:
    """Map an ENA flatfile frame into ``VIRUS_RAW_COLUMNS``."""
    if df.empty:
        return empty_virus_frame()

    rows: list[dict[str, str]] = []
    for _, raw in df.iterrows():
        country_raw = str(raw.get("country", "") or "").strip()
        country_field, subnational = split_country_subnational(country_raw)

        existing_lat = str(raw.get("lat", "") or "").strip()
        existing_lon = str(raw.get("lon", "") or "").strip()
        existing_source = str(raw.get("coord_source", "") or "").strip()
        location = str(raw.get("location", "") or "").strip()

        if existing_lat and existing_lon and existing_source in {"exact_gps", "sample_backfill"}:
            lat, lon, coord_source = existing_lat, existing_lon, existing_source
        else:
            lat, lon, coord_source = resolve_coords(
                lat=existing_lat,
                lon=existing_lon,
                location=location,
                coord_source="exact_gps",
            )

        accession_value = normalize_accession(raw.get(accession_column, ""))
        sample_accession = str(raw.get("sample_accession", "") or "").strip()
        native = accession_value or sample_accession or str(raw.get("run_accession", "") or "").strip()

        rows.append(
            {
                "record_id": f"{source}:{native}" if native else source,
                "source": source,
                "accession": accession_value,
                "sample_accession": sample_accession,
                "study_accession": str(raw.get("study_accession", "") or "").strip(),
                "pathogen_name": str(raw.get("pathogen_name", "") or "").strip(),
                "vector_group": str(raw.get("vector_group", "") or "").strip(),
                "query_tax_id": str(raw.get("query_tax_id", "") or "").strip(),
                "scientific_name": str(raw.get("scientific_name", "") or "").strip(),
                "tax_id": str(raw.get("tax_id", "") or "").strip(),
                "country": country_field,
                "subnational_region": subnational,
                "nuts_id": "",
                "lat": lat,
                "lon": lon,
                "coord_source": coord_source,
                "collection_date": str(raw.get("collection_date", "") or "").strip(),
                "host": str(raw.get("host", "") or "").strip(),
                "isolation_source": str(raw.get("isolation_source", "") or "").strip(),
            }
        )

    return ensure_virus_columns(pd.DataFrame(rows))


print("ENA normalize / backfill helpers ready")

ENA normalize / backfill helpers ready


In [5]:
sequence_frames: list[pd.DataFrame] = []
read_run_frames: list[pd.DataFrame] = []
ena_per_pathogen: dict[str, dict] = {}

for target in tqdm(TARGETS, desc="ENA pathogens"):
    pathogen = target["pathogen"]
    vector_group = target["vector_group"]
    seq_parts: list[pd.DataFrame] = []
    run_parts: list[pd.DataFrame] = []
    raw_sequence = 0
    raw_read_run = 0

    for tax_id in target["tax_ids"]:
        query = f"tax_tree({tax_id})"

        seq = ena_search_tsv(
            result="sequence", query=query, fields=ENA_SEQUENCE_FIELDS
        )
        if not seq.empty:
            seq = seq.copy()
            seq["query_tax_id"] = str(tax_id)
            seq["pathogen_name"] = pathogen
            seq["vector_group"] = vector_group
            seq_parts.append(seq)
            raw_sequence += len(seq)

        runs = ena_search_tsv(
            result="read_run", query=query, fields=ENA_READ_RUN_FIELDS
        )
        if not runs.empty:
            runs = runs.copy()
            runs["query_tax_id"] = str(tax_id)
            runs["pathogen_name"] = pathogen
            runs["vector_group"] = vector_group
            run_parts.append(runs)
            raw_read_run += len(runs)

    if seq_parts:
        sequence_frames.append(pd.concat(seq_parts, ignore_index=True))
    if run_parts:
        read_run_frames.append(pd.concat(run_parts, ignore_index=True))

    ena_per_pathogen[pathogen] = {
        "tax_ids": list(target["tax_ids"]),
        "raw_sequence": raw_sequence,
        "raw_read_run": raw_read_run,
    }

raw_sequence_df = (
    pd.concat(sequence_frames, ignore_index=True)
    if sequence_frames
    else pd.DataFrame(columns=[*ENA_SEQUENCE_FIELDS, "query_tax_id", "pathogen_name", "vector_group"])
)
raw_read_run_df = (
    pd.concat(read_run_frames, ignore_index=True)
    if read_run_frames
    else pd.DataFrame(columns=[*ENA_READ_RUN_FIELDS, "query_tax_id", "pathogen_name", "vector_group"])
)

# Drop exact duplicate query hits within each result type before sample backfill.
if not raw_sequence_df.empty:
    raw_sequence_df = raw_sequence_df.drop_duplicates(
        subset=["accession", "pathogen_name"], keep="first"
    ).reset_index(drop=True)
if not raw_read_run_df.empty:
    raw_read_run_df = raw_read_run_df.drop_duplicates(
        subset=["run_accession", "pathogen_name"], keep="first"
    ).reset_index(drop=True)

print(f"Raw ENA sequence rows: {len(raw_sequence_df):,}")
print(f"Raw ENA read_run rows: {len(raw_read_run_df):,}")


ENA pathogens:   0%|          | 0/26 [00:00<?, ?it/s]


ENA pathogens:   4%|▍         | 1/26 [01:04<27:03, 64.94s/it]


ENA pathogens:   8%|▊         | 2/26 [01:05<10:51, 27.16s/it]


ENA pathogens:  12%|█▏        | 3/26 [01:20<08:10, 21.34s/it]


ENA pathogens:  15%|█▌        | 4/26 [01:33<06:43, 18.35s/it]


ENA pathogens:  19%|█▉        | 5/26 [01:48<05:55, 16.94s/it]

... (19 outputs left out)


ENA pathogens:  96%|█████████▌| 25/26 [02:53<00:03,  3.65s/it]


ENA pathogens: 100%|██████████| 26/26 [02:55<00:00,  3.33s/it]


ENA pathogens: 100%|██████████| 26/26 [02:55<00:00,  6.77s/it]

Raw ENA sequence rows: 144,009
Raw ENA read_run rows: 20,671


In [6]:
sample_ids = sorted(
    {
        *(raw_sequence_df.get("sample_accession", pd.Series(dtype=str)).fillna("").astype(str).str.strip()),
        *(raw_read_run_df.get("sample_accession", pd.Series(dtype=str)).fillna("").astype(str).str.strip()),
    }
    - {""}
)
print(f"Distinct sample_accession values: {len(sample_ids):,}")

sample_df = ena_sample_by_accessions(sample_ids)
print(f"BioSample rows returned: {len(sample_df):,}")

# Seed lat/lon/coord_source on sequence rows from the location string first.
if not raw_sequence_df.empty:
    seq_coords = [
        resolve_coords(location=loc, coord_source="exact_gps")
        for loc in raw_sequence_df.get("location", pd.Series("", index=raw_sequence_df.index))
    ]
    raw_sequence_df = raw_sequence_df.copy()
    raw_sequence_df["lat"] = [c[0] for c in seq_coords]
    raw_sequence_df["lon"] = [c[1] for c in seq_coords]
    raw_sequence_df["coord_source"] = [c[2] for c in seq_coords]

if not raw_read_run_df.empty:
    raw_read_run_df = raw_read_run_df.copy()
    run_coords = [
        resolve_coords(
            lat=lat,
            lon=lon,
            location=loc,
            coord_source="exact_gps",
        )
        for lat, lon, loc in zip(
            raw_read_run_df.get("lat", pd.Series("", index=raw_read_run_df.index)),
            raw_read_run_df.get("lon", pd.Series("", index=raw_read_run_df.index)),
            raw_read_run_df.get("location", pd.Series("", index=raw_read_run_df.index)),
            strict=True,
        )
    ]
    raw_read_run_df["lat"] = [c[0] for c in run_coords]
    raw_read_run_df["lon"] = [c[1] for c in run_coords]
    raw_read_run_df["coord_source"] = [c[2] for c in run_coords]

sequence_bf = backfill_from_samples(raw_sequence_df, sample_df)
read_run_bf = backfill_from_samples(raw_read_run_df, sample_df)

ena_sequence_df = apply_keep_filter(
    normalize_ena_rows(sequence_bf, source="ena_sequence", accession_column="accession")
)
ena_read_run_df = apply_keep_filter(
    normalize_ena_rows(read_run_bf, source="ena_read_run", accession_column="run_accession")
)

# Keep run accessions in record_id only — blank the sequence accession field so later
# INSDC merges group on true assembled accessions / sample_accession.
if not ena_read_run_df.empty:
    ena_read_run_df = ena_read_run_df.copy()
    ena_read_run_df["accession"] = ""

ena_df = ensure_virus_columns(
    pd.concat([ena_sequence_df, ena_read_run_df], ignore_index=True)
    if len(ena_sequence_df) or len(ena_read_run_df)
    else empty_virus_frame()
)
ena_df = assign_record_ids(ena_df)

for pathogen, stats in ena_per_pathogen.items():
    stats["kept_sequence"] = int((ena_sequence_df["pathogen_name"] == pathogen).sum()) if len(ena_sequence_df) else 0
    stats["kept_read_run"] = int((ena_read_run_df["pathogen_name"] == pathogen).sum()) if len(ena_read_run_df) else 0
    stats["kept_total"] = stats["kept_sequence"] + stats["kept_read_run"]

source_frames["ena"] = ena_df
source_stats["ena"] = {
    "sources": ["ena_sequence", "ena_read_run"],
    "citation": ENA_CITATION,
    "endpoint": ENA_SEARCH_URL,
    "raw_sequence": int(len(raw_sequence_df)),
    "raw_read_run": int(len(raw_read_run_df)),
    "biosamples_requested": len(sample_ids),
    "biosamples_returned": int(len(sample_df)),
    "kept_sequence": int(len(ena_sequence_df)),
    "kept_read_run": int(len(ena_read_run_df)),
    "kept_total": int(len(ena_df)),
    "coord_source_counts": ena_df["coord_source"].value_counts(dropna=False).to_dict(),
    "per_pathogen": ena_per_pathogen,
}

print(f"ENA kept sequence rows: {len(ena_sequence_df):,}")
print(f"ENA kept read_run rows:  {len(ena_read_run_df):,}")
print(f"ENA combined rows:       {len(ena_df):,}")
print("coord_source counts:")
print(ena_df["coord_source"].value_counts(dropna=False).to_string())
pd.DataFrame(ena_per_pathogen).T

Distinct sample_accession values: 16,125



ENA samples:   0%|          | 0/17 [00:00<?, ?it/s]


ENA samples:   6%|▌         | 1/17 [00:04<01:07,  4.22s/it]


ENA samples:  12%|█▏        | 2/17 [00:04<00:30,  2.05s/it]


ENA samples:  18%|█▊        | 3/17 [00:05<00:18,  1.31s/it]


ENA samples:  24%|██▎       | 4/17 [00:06<00:16,  1.29s/it]

... (12 outputs left out)


ENA samples: 100%|██████████| 17/17 [00:25<00:00,  1.13s/it]

BioSample rows returned: 15,897


ENA kept sequence rows: 118,384
ENA kept read_run rows:  17,111
ENA combined rows:       135,495
coord_source counts:
coord_source
none               125115
exact_gps            8189
sample_backfill      2191


,tax_ids,raw_sequence,raw_read_run,kept_sequence,kept_read_run,kept_total
West Nile virus,[3048448],25374,4132,8519,4095,12614
Usutu virus,[3048443],1995,418,1905,403,2308
Dengue virus,[3052464],60229,8376,57253,5225,62478
Chikungunya virus,[3426298],11178,2122,10030,2101,12131
Zika virus,[3048459],3378,3963,2245,3785,6030
Sindbis virus,[3426318],820,525,515,464,979
Rift Valley fever virus,[3052676],1912,140,1760,140,1900
Batai orthobunyavirus,[3052371],126,0,120,0,120
Tahyna / California encephalitis orthobunyavirus,"[3052445, 35305, 178787]",139,1,130,0,130
Inkoo orthobunyavirus,[45269],31,0,25,0,25


## BV-BRC — Genome REST API

1. Query `https://www.bv-brc.org/api/genome/` per pathogen with `in(taxon_lineage_ids,(...))` over ICTV + legacy tax IDs (`lineage_tax_ids`).
2. Page with `limit(25000, offset)` and `Accept: text/tsv` (`Content-Range` total).
3. Map GenBank / BioSample / BioProject / place / time / host fields; prefer an accession already seen in ENA via `pick_primary_accession`.
4. Tag `source=bvbrc`, apply the place/time keep filter, register `source_frames["bvbrc"]`.

Output: `bvbrc_df`.

In [ ]:
_CONTENT_RANGE_RE = re.compile(r"items\s+(\d+)-(\d+)/(\d+|\*)", re.IGNORECASE)


def _http_body_and_headers(
    request: Request,
    *,
    timeout_s: int = DOWNLOAD_TIMEOUT_S,
    retries: int = MAX_RETRIES,
) -> tuple[bytes, dict[str, str]]:
    last_error: Exception | None = None
    for attempt in range(retries):
        try:
            with urlopen(request, timeout=timeout_s) as response:
                headers = {k: v for k, v in response.headers.items()}
                chunks: list[bytes] = []
                while True:
                    chunk = response.read(CHUNK_SIZE)
                    if not chunk:
                        break
                    chunks.append(chunk)
            return b"".join(chunks), headers
        except HTTPError as exc:
            last_error = exc
            if 400 <= int(exc.code) < 500:
                break
            if attempt + 1 >= retries:
                break
            time.sleep(RETRY_BACKOFF_S * (2**attempt))
        except (URLError, TimeoutError) as exc:
            last_error = exc
            if attempt + 1 >= retries:
                break
            time.sleep(RETRY_BACKOFF_S * (2**attempt))
    assert last_error is not None
    raise last_error


def _parse_content_range_total(headers: dict[str, str]) -> int | None:
    raw = headers.get("Content-Range") or headers.get("content-range") or ""
    match = _CONTENT_RANGE_RE.search(raw)
    if not match:
        return None
    total = match.group(3)
    if total == "*":
        return None
    return int(total)


def bvbrc_genome_page(
    *,
    lineage_ids: list[int] | tuple[int, ...],
    fields: list[str] = BVBRC_SELECT_FIELDS,
    limit: int = BVBRC_PAGE_SIZE,
    offset: int = 0,
    timeout_s: int = DOWNLOAD_TIMEOUT_S,
) -> tuple[pd.DataFrame, int | None]:
    """Fetch one BV-BRC genome TSV page; return ``(frame, total_or_None)``."""
    ids = ",".join(str(tax_id) for tax_id in lineage_ids)
    select = ",".join(fields)
    url = (
        f"{BVBRC_GENOME_URL}"
        f"?limit({int(limit)},{int(offset)})"
        f"&in(taxon_lineage_ids,({ids}))"
        f"&select({select})"
    )
    request = Request(
        url,
        headers={
            "User-Agent": USER_AGENT,
            "Accept": "text/tsv",
        },
    )
    body, headers = _http_body_and_headers(request, timeout_s=timeout_s)
    text = body.decode("utf-8", errors="replace")
    total = _parse_content_range_total(headers)
    if not text.strip():
        return pd.DataFrame(columns=fields), total
    frame = pd.read_csv(io.StringIO(text), sep="\t", dtype=str, keep_default_na=False)
    return frame, total


def fetch_bvbrc_genomes_for_target(
    target: dict,
    *,
    page_size: int = BVBRC_PAGE_SIZE,
) -> pd.DataFrame:
    """Page through all BV-BRC genomes for one pathogen lineage union."""
    lineage_ids = lineage_tax_ids(target)
    frames: list[pd.DataFrame] = []
    offset = 0
    total: int | None = None

    while True:
        page, page_total = bvbrc_genome_page(
            lineage_ids=lineage_ids,
            limit=page_size,
            offset=offset,
        )
        if page_total is not None:
            total = page_total
        if page.empty:
            break
        page = page.copy()
        page["query_tax_id"] = str(target["tax_ids"][0])
        page["pathogen_name"] = target["pathogen"]
        page["vector_group"] = target["vector_group"]
        frames.append(page)
        offset += len(page)
        if total is not None and offset >= total:
            break
        if len(page) < page_size:
            break

    if not frames:
        return pd.DataFrame(
            columns=[*BVBRC_SELECT_FIELDS, "query_tax_id", "pathogen_name", "vector_group"]
        )
    return pd.concat(frames, ignore_index=True)


def normalize_bvbrc_rows(
    df: pd.DataFrame,
    *,
    known_accessions: set[str] | None = None,
) -> pd.DataFrame:
    """Map BV-BRC genome rows into ``VIRUS_RAW_COLUMNS`` (``source=bvbrc``)."""
    if df.empty:
        return empty_virus_frame()

    rows: list[dict[str, str]] = []
    for _, raw in df.iterrows():
        genome_id = str(raw.get("genome_id", "") or "").strip()
        accession = pick_primary_accession(
            raw.get("genbank_accessions", ""),
            known_accessions=known_accessions,
        )
        sample_accession = str(raw.get("biosample_accession", "") or "").strip()
        study_accession = str(raw.get("bioproject_accession", "") or "").strip()

        geographic_location = str(raw.get("geographic_location", "") or "").strip()
        isolation_country = str(raw.get("isolation_country", "") or "").strip()
        state_province = str(raw.get("state_province", "") or "").strip()

        if geographic_location:
            country_field, subnational = split_country_subnational(geographic_location)
        elif isolation_country:
            country_field, subnational = split_country_subnational(isolation_country)
        else:
            country_field, subnational = "", ""

        if not country_field and isolation_country:
            country_field = isolation_country
        if not subnational and state_province:
            subnational = state_province

        lat, lon, coord_source = resolve_coords(
            lat=raw.get("latitude", ""),
            lon=raw.get("longitude", ""),
            coord_source="exact_gps",
        )

        collection_date = str(raw.get("collection_date", "") or "").strip()
        if not collection_date:
            collection_date = str(raw.get("collection_year", "") or "").strip()

        native = accession or genome_id or sample_accession
        rows.append(
            {
                "record_id": f"bvbrc:{native}" if native else "bvbrc",
                "source": "bvbrc",
                "accession": accession,
                "sample_accession": sample_accession,
                "study_accession": study_accession,
                "pathogen_name": str(raw.get("pathogen_name", "") or "").strip(),
                "vector_group": str(raw.get("vector_group", "") or "").strip(),
                "query_tax_id": str(raw.get("query_tax_id", "") or "").strip(),
                "scientific_name": str(raw.get("genome_name", "") or "").strip(),
                "tax_id": str(raw.get("taxon_id", "") or "").strip(),
                "country": country_field,
                "subnational_region": subnational,
                "nuts_id": "",
                "lat": lat,
                "lon": lon,
                "coord_source": coord_source,
                "collection_date": collection_date,
                "host": str(raw.get("host_name", "") or "").strip(),
                "isolation_source": str(raw.get("isolation_source", "") or "").strip(),
            }
        )

    return ensure_virus_columns(pd.DataFrame(rows))


print("BV-BRC helpers ready")

In [ ]:
known_ena_accessions: set[str] = set()
if "ena" in source_frames and not source_frames["ena"].empty:
    known_ena_accessions = {
        acc
        for acc in (
            normalize_accession(value) for value in source_frames["ena"]["accession"]
        )
        if acc
    }
print(f"Known ENA accessions for BV-BRC primary pick: {len(known_ena_accessions):,}")

bvbrc_raw_frames: list[pd.DataFrame] = []
bvbrc_per_pathogen: dict[str, dict] = {}

for target in tqdm(TARGETS, desc="BV-BRC pathogens"):
    pathogen = target["pathogen"]
    raw = fetch_bvbrc_genomes_for_target(target)
    if not raw.empty:
        raw = raw.drop_duplicates(subset=["genome_id", "pathogen_name"], keep="first")
        bvbrc_raw_frames.append(raw)
    bvbrc_per_pathogen[pathogen] = {
        "tax_ids": list(target["tax_ids"]),
        "legacy_tax_ids": list(target.get("legacy_tax_ids", ())),
        "lineage_tax_ids": list(lineage_tax_ids(target)),
        "raw": int(len(raw)),
    }

raw_bvbrc_df = (
    pd.concat(bvbrc_raw_frames, ignore_index=True)
    if bvbrc_raw_frames
    else pd.DataFrame(
        columns=[*BVBRC_SELECT_FIELDS, "query_tax_id", "pathogen_name", "vector_group"]
    )
)

bvbrc_df = apply_keep_filter(
    normalize_bvbrc_rows(raw_bvbrc_df, known_accessions=known_ena_accessions)
)
bvbrc_df = assign_record_ids(bvbrc_df)

for pathogen, stats in bvbrc_per_pathogen.items():
    stats["kept"] = (
        int((bvbrc_df["pathogen_name"] == pathogen).sum()) if len(bvbrc_df) else 0
    )

source_frames["bvbrc"] = bvbrc_df
source_stats["bvbrc"] = {
    "sources": ["bvbrc"],
    "citation": BVBRC_CITATION,
    "endpoint": BVBRC_GENOME_URL,
    "raw": int(len(raw_bvbrc_df)),
    "kept": int(len(bvbrc_df)),
    "known_ena_accessions": len(known_ena_accessions),
    "matched_ena_accession": int(
        bvbrc_df["accession"].isin(known_ena_accessions).sum()
    )
    if len(bvbrc_df) and known_ena_accessions
    else 0,
    "coord_source_counts": bvbrc_df["coord_source"].value_counts(dropna=False).to_dict(),
    "per_pathogen": bvbrc_per_pathogen,
}

print(f"BV-BRC raw genome rows: {len(raw_bvbrc_df):,}")
print(f"BV-BRC kept rows:       {len(bvbrc_df):,}")
print("coord_source counts:")
print(bvbrc_df["coord_source"].value_counts(dropna=False).to_string())
pd.DataFrame(
    {
        pathogen: {
            "raw": stats["raw"],
            "kept": stats["kept"],
        }
        for pathogen, stats in bvbrc_per_pathogen.items()
    }
).T

## Other sources (pending)

Subsequent tasks add Pathoplexus, FAO EMPRES-i, and ECDC NUTS3 sections that append to `source_frames` / `source_stats`. Full INSDC + outbreak dedupe lands in the final merge task.

## Interim write (ENA + BV-BRC)

Until the remaining feeds and the final multi-source merge/dedupe land, concatenate the implemented source frames into a schema-compatible extract. Overlapping INSDC accessions may appear in both ENA and BV-BRC rows until the merge task.

In [7]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(CHUNK_SIZE), b""):
            digest.update(chunk)
    return digest.hexdigest()


interim_parts = [
    ensure_virus_columns(source_frames[key])
    for key in ("ena", "bvbrc")
    if key in source_frames and not source_frames[key].empty
]
df = (
    ensure_virus_columns(pd.concat(interim_parts, ignore_index=True))
    if interim_parts
    else empty_virus_frame()
)
df.to_csv(TSV_PATH, sep="\t", index=False)

manifest = {
    "fetched_at": datetime.now(timezone.utc).isoformat(),
    "tsv_name": TSV_NAME,
    "tsv_bytes": TSV_PATH.stat().st_size,
    "sha256": sha256_file(TSV_PATH),
    "row_count": len(df),
    "columns": list(VIRUS_RAW_COLUMNS),
    "filter": (
        "keep rows with non-empty country, nuts_id, (lat,lon), or collection_date; "
        "ENA sample backfill applied before keep filter; BV-BRC included; "
        "full multi-source dedupe pending"
    ),
    "sources": source_stats,
    "coord_source_counts": df["coord_source"].value_counts(dropna=False).to_dict(),
    "per_pathogen": {
        pathogen: int((df["pathogen_name"] == pathogen).sum())
        for pathogen in sorted(df["pathogen_name"].unique())
    },
    "targets": [
        {
            "pathogen": t["pathogen"],
            "tax_ids": list(t["tax_ids"]),
            "legacy_tax_ids": list(t.get("legacy_tax_ids", ())),
            "vector_group": t["vector_group"],
        }
        for t in TARGETS
    ],
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")

print(f"Wrote {TSV_PATH} ({manifest['tsv_bytes']:,} bytes)")
print(f"Wrote {MANIFEST_PATH}")
print("Rows by source:")
print(df["source"].value_counts(dropna=False).to_string())
manifest["sha256"], manifest["row_count"]

Wrote /workspaces/vector-guard/data/raw/virus/virus_detections.tsv (22,367,100 bytes)
Wrote /workspaces/vector-guard/data/raw/virus/manifest.json


('de241fd1fdf18c86a0e6618b8c30cc06184fffcce6d8e68577c46113ee0a9406', 135495)

## Smoke check

Reload via `load_virus_detections` and confirm the unified schema plus ENA / BV-BRC source tags.

In [8]:
print("Files in raw dir:")
for rel in sorted(
    p.relative_to(RAW_DIR).as_posix() for p in RAW_DIR.rglob("*") if p.is_file()
):
    print(f"  - {rel}")

core_path = resolve_virus_core_path(RAW_DIR)
loaded = load_virus_detections(RAW_DIR)
print(f"\nCore file: {core_path.name}")
print(f"Rows: {len(loaded):,}")
print(f"Columns ({len(loaded.columns)}): {list(loaded.columns)}")
missing_cols = [c for c in VIRUS_RAW_COLUMNS if c not in loaded.columns]
print(f"Missing canonical columns: {missing_cols or 'none'}")
print("\nRows by source:")
print(loaded["source"].value_counts().to_string())
print("\nRows by coord_source:")
print(loaded["coord_source"].value_counts().to_string())
print("\nRows by pathogen_name:")
print(loaded["pathogen_name"].value_counts().to_string())
print("\nRows by vector_group:")
print(loaded["vector_group"].value_counts().to_string())
loaded.head(8).T

Files in raw dir:
  - .gitkeep
  - manifest.json
  - virus_detections.tsv



Core file: virus_detections.tsv
Rows: 135,495
Columns (19): ['record_id', 'source', 'accession', 'sample_accession', 'study_accession', 'pathogen_name', 'vector_group', 'query_tax_id', 'scientific_name', 'tax_id', 'country', 'subnational_region', 'nuts_id', 'lat', 'lon', 'coord_source', 'collection_date', 'host', 'isolation_source']
Missing canonical columns: none

Rows by source:
source
ena_sequence    118384
ena_read_run     17111

Rows by coord_source:
coord_source
none               125115
exact_gps            8189
sample_backfill      2191

Rows by pathogen_name:
pathogen_name
Dengue virus                                         62478
Bluetongue virus                                     14179
West Nile virus                                      12614
Chikungunya virus                                    12131
Zika virus                                            6030
Crimean-Congo hemorrhagic fever virus                 5416
Tick-borne encephalitis virus                         35

,0,1,2,3,4,5,6,7
record_id,ena_sequence:MH170264,ena_sequence:MH819470,ena_sequence:MW803141,ena_sequence:MW835355,ena_sequence:MW835358,ena_sequence:MW862090,ena_sequence:MW862097,ena_sequence:MW862099
source,ena_sequence,ena_sequence,ena_sequence,ena_sequence,ena_sequence,ena_sequence,ena_sequence,ena_sequence
accession,MH170264,MH819470,MW803141,MW835355,MW835358,MW862090,MW862097,MW862099
sample_accession,,,,,,,,
study_accession,,PRJNA480662,,,,,,
pathogen_name,West Nile virus,West Nile virus,West Nile virus,West Nile virus,West Nile virus,West Nile virus,West Nile virus,West Nile virus
vector_group,mosquito,mosquito,mosquito,mosquito,mosquito,mosquito,mosquito,mosquito
query_tax_id,3048448,3048448,3048448,3048448,3048448,3048448,3048448,3048448
scientific_name,West Nile virus,West Nile virus,West Nile virus,West Nile virus,West Nile virus,West Nile virus,West Nile virus,West Nile virus
tax_id,11082,11082,11082,11082,11082,11082,11082,11082
